# Vérifier la conformité à l’AI Act européen avec Gemini et Agent Module


> Ce notebook a été contribué par [Agent Module](https://agent-module.dev/) et adapté en français. Il appelle un service tiers : vérifiez ses conditions d’utilisation, ses limites et le traitement des données avant d’y envoyer des informations réelles.


L’[AI Act de l’Union européenne](https://artificialintelligenceact.eu/) s’applique progressivement à partir d’août 2026. Toute organisation qui développe, déploie ou utilise des systèmes d’IA sur le marché européen doit pouvoir démontrer une démarche de conformité.

[Agent Module](https://agent-module.dev) fournit une logique de conformité structurée et déterministe, récupérable au moment de l’exécution par un agent. Chaque module de conformité IA est associé à des articles précis de l’AI Act, à des règles binaires de réussite/échec, à des déclencheurs d’escalade et à des références réglementaires.

Dans ce notebook, vous allez :

1. Créer des fonctions Python qui encapsulent l’API REST d’Agent Module.
2. Tester chaque fonction isolément.
3. Transmettre ces fonctions à Gemini via le [function calling](https://ai.google.dev/gemini-api/docs/function-calling).
4. Laisser Gemini récupérer une logique de conformité et l’appliquer à un scénario.

> **Limite importante :** il s’agit d’une démonstration pédagogique de pré-analyse. Le résultat ne constitue ni un avis juridique ni une certification de conformité.


## Installation et configuration


### Installer les dépendances


In [1]:
# Installation silencieuse des dépendances requises :
# - google-genai : le SDK officiel de Google pour interagir avec Gemini
# - requests : la bibliothèque standard pour envoyer des requêtes HTTP à l'API d'Agent Module
%pip install -qU google-genai requests

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 263.5/263.5 kB 10.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.59.1 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


### Configurer la clé API

Dans Colab, ajoutez votre clé Gemini dans un secret nommé `GEMINI_API_KEY` puis autorisez le notebook à le lire. Ne placez jamais une clé API directement dans une cellule partagée.


In [3]:
import json
import os
import uuid
from getpass import getpass

import requests
from google import genai

# Tentative de récupération sécurisée de la clé d'API Gemini depuis le gestionnaire de secrets de Colab
try:
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
except Exception:
    # Alternativement, récupération depuis les variables d'environnement système
    GEMINI_API_KEY = os.environ.get('GEMINI_API_KEY')

# Si aucune clé n'est trouvée, demande manuelle interactive via une invite de saisie sécurisée
if not GEMINI_API_KEY:
    GEMINI_API_KEY = getpass('Saisissez votre clé GEMINI_API_KEY : ')
if not GEMINI_API_KEY:
    raise ValueError('Clé API absente.')

# Initialisation du client d'API Gemini avec la clé configurée
client = genai.Client(api_key=GEMINI_API_KEY)

### Choisir un modèle

Le function calling fonctionne avec les modèles Gemini compatibles. Pour cette analyse et la démonstration agentique, le modèle GA `gemini-3.8-flash` constitue le choix principal. Pour réduire coût et latence en classe, vous pouvez choisir `gemini-3.5-flash-lite`, au prix d’une capacité de raisonnement généralement plus limitée.


In [14]:
# Définition du modèle Gemini à utiliser.
# 'gemini-3.5-flash-lite' est idéal pour la rapidité et le faible coût.
# 'gemini-3.8-flash' peut être choisi pour de meilleures capacités de raisonnement.
MODEL_ID = 'gemini-3.5-flash-lite'
print(f'Modèle utilisé : {MODEL_ID}')

Modèle utilisé : gemini-3.5-flash-lite


## Vérifier l’API Agent Module

Avant de créer les outils, vérifiez que le service Agent Module est joignable et examinez le nombre de modules de conformité disponibles. Le nombre est interrogé dans le service au moment de l’exécution ; il peut donc évoluer.


In [5]:
# URL de base pour interagir avec les services d'Agent Module
AGENT_MODULE_API = "https://api.agent-module.dev"

# Appel de diagnostic pour s'assurer que le service tiers est en ligne
status = requests.get(f"{AGENT_MODULE_API}/api/status", timeout=10).json()

# Extraction du nombre de modules actifs dédiés à la conformité de l'IA (AI Act)
ai_compliance_module_count = status["ai_compliance"]["node_count"]

# Affichage des informations opérationnelles récupérées en temps réel
print(f"État de l’API :             {status['api_status']}")
print(f"Modules de conformité IA :  {ai_compliance_module_count}")
print(f"Cohorte active :          {status['active_cohort']}")

État de l’API :             operational
Modules de conformité IA :  23
Cohorte active :          inaugural


## Définir les outils Agent Module

Nous encapsulons le [point d’accès MCP d’Agent Module](https://github.com/AgentModule/mcp) dans des fonctions Python que Gemini peut appeler. Le SDK produit automatiquement les déclarations de fonctions à partir des annotations de type et des docstrings : rédigez-les donc clairement.

Commençons par une petite fonction de transport JSON-RPC.


In [6]:
# Point d'accès (endpoint) pour le protocole de communication MCP (Model Context Protocol)
AGENT_MODULE_MCP = f"{AGENT_MODULE_API}/mcp"


def _mcp_call(tool_name: str, arguments: dict) -> dict:
    """Envoie une requête JSON-RPC au point d'accès MCP d’Agent Module.

    Cette fonction utilitaire interne encapsule la structure formelle de requête
    JSON-RPC pour simplifier les appels vers les différents outils d'Agent Module.
    """
    # Préparation de l'enveloppe JSON-RPC 2.0
    payload = {
        "jsonrpc": "2.0",
        "method": "tools/call",
        "params": {"name": tool_name, "arguments": arguments},
        "id": 1,
    }
    try:
        # Transmission de la requête HTTP POST avec un délai maximal de 30 secondes
        response = requests.post(
            AGENT_MODULE_MCP,
            headers={"Content-Type": "application/json"},
            json=payload,
            timeout=30,
        )
        response.raise_for_status() # Déclenche une erreur si le code HTTP n'est pas un succès
        return response.json()
    except requests.exceptions.Timeout:
        return {"error": "Délai dépassé : Agent Module peut être temporairement indisponible."}
    except requests.exceptions.HTTPError as e:
        return {"error": f"Erreur HTTP : {e.response.status_code}"}
    except requests.exceptions.RequestException as e:
        return {"error": f"Échec de connexion : {str(e)}"}

### Obtenir une clé d’essai

Définissez le premier outil : il demande une clé d’essai gratuite valable 24 heures. Elle permet l’accès aux modules de conformité pendant l’atelier. Cette offre et ses limites peuvent évoluer : vérifiez-les auprès du service.


In [7]:
# Génération d'un identifiant unique de session pour éviter les collisions de clés d'essai
_session_agent_id = f"gemini-cookbook-{uuid.uuid4().hex[:8]}"

# Variable globale temporaire pour stocker la clé d'essai obtenue et éviter d'en générer plusieurs
_trial_key = None


def get_trial_key(agent_id: str) -> dict:
    """Obtient une clé d’essai Agent Module valable 24 h pour la logique de conformité AI Act.

    Appelez cette fonction une fois au début de la session. La clé est valable 24 heures
    et inclut 500 appels API gratuits. Les appels suivants réutilisent la clé
    stockée dans cette session au lieu d’en demander une nouvelle.

    Args:
        agent_id: Identifiant stable de votre agent, par exemple 'gemini-cookbook-demo'.
    """
    global _trial_key

    # Mécanisme d'idempotence : si une clé est déjà chargée en mémoire, on la retourne directement
    if _trial_key is not None:
        return {
            "status": "reused",
            "trial_key": _trial_key,
            "note": "Utilisation de la clé d’essai existante dans cette session.",
        }

    # Appel au service MCP pour demander une clé d'essai
    result = _mcp_call("get_trial_key", {"agent_id": agent_id})

    # Analyse de la réponse et extraction de la clé d'essai si disponible
    if "result" in result:
        content = result["result"]
        if isinstance(content, dict) and "content" in content:
            for item in content["content"]:
                if item.get("type") == "text":
                    data = json.loads(item["text"])
                    # Gestion de l'erreur si une clé d'essai est déjà active pour cet ID
                    if data.get("status") == "already_active":
                        return {
                            "error": "already_active",
                            "message": (
                                f"Un essai actif existe déjà pour agent_id="
                                f"{agent_id!r}. Attendez 24 heures ou modifiez l’"
                                "agent_id, puis relancez."
                            ),
                        }
                    if "trial_key" in data:
                        _trial_key = data["trial_key"]
    return result

Testez l’obtention de clé afin de vérifier que le service fonctionne.


In [8]:
# Exécution du test pour acquérir la clé d'essai pour la session courante
result = get_trial_key(_session_agent_id)

# Validation du résultat et affichage des détails
print(f'agent_id de session : {_session_agent_id}')
print(f'Clé d’essai obtenue : {_trial_key is not None}')
print(f'Préfixe de clé : {_trial_key[:12]}...' if _trial_key else 'Aucune clé émise')

agent_id de session : gemini-cookbook-9796d689
Clé d’essai obtenue : True
Préfixe de clé : am_trial_841...


### Récupérer une logique de conformité

Le second outil récupère la logique structurée d’un module de conformité particulier : règles déterministes, références réglementaires, conditions de réussite/échec et déclencheurs d’escalade.


In [9]:
def retrieve_compliance_logic(vertical: str, node_id: str) -> dict:
    """Récupère une logique de conformité structurée à l’AI Act auprès d’Agent Module.

    Renvoie des règles déterministes, des références réglementaires, des conditions
    de réussite/échec et des déclencheurs d’escalade pour un module donné.

    Args:
        vertical: Verticale de connaissances. Utilisez 'ai-compliance' (ou l’alias historique 'ethics')
            pour les modules AI Act.
        node_id: Nœud à récupérer. Format : 'node:ethics:eth{NNN}:logic'.
            Exemple : 'node:ethics:eth001:logic' pour la souveraineté des données,
            'node:ethics:eth003:logic' pour la transparence.
    """
    # Préparation des paramètres de requête requis par l'API MCP
    arguments = {"vertical": vertical, "node": node_id}

    # Ajout du jeton d'essai s'il a été généré correctement au préalable
    if _trial_key:
        arguments["token"] = _trial_key

    # Transmission de la requête de récupération de la base de connaissances
    return _mcp_call("query_knowledge", arguments)

Testez la récupération du module de souveraineté des données, ETH_001, associé notamment aux articles 9 et 10 de l’AI Act.


In [10]:
# Test unitaire : Récupération directe des règles du module de Souveraineté des Données (ETH_001)
result = retrieve_compliance_logic("ethics", "node:ethics:eth001:logic")

# Affichage des 2000 premiers caractères du fichier JSON reçu pour en vérifier la structure
print(json.dumps(result, indent=4)[:2000])

{
    "jsonrpc": "2.0",
    "id": 1,
    "result": {
        "content": [
            {
                "type": "text",
                "text": "{\n  \"object\": \"traversal_response\",\n  \"vertical\": \"ethics\",\n  \"node_id\": \"node:ethics:eth001:logic\",\n  \"layer\": \"logic\",\n  \"access\": \"trial\",\n  \"content\": {\n    \"node_id\": \"ETH_001_SOVEREIGNTY_logic\",\n    \"vertical\": \"ethics\",\n    \"version\": \"2.0.0\",\n    \"last_validated\": \"2026-02-28\",\n    \"wikidata_ref\": \"https://www.wikidata.org/wiki/Q1066186\",\n    \"eu_ai_act_articles\": [\n      \"Art. 10\",\n      \"Art. 13\"\n    ],\n    \"weighting\": \"Essential\",\n    \"subject\": \"sovereignty_and_data_ownership\",\n    \"description\": \"Deterministic rules governing user data sovereignty, ownership rights, consent granularity, and data governance obligations under GDPR and EU AI Act Art. 10. The user retains absolute ownership of all data processed by the agent \u2014 the agent is a custodian, 

### Envoyer une évaluation de valeur

Le troisième outil envoie une évaluation de la qualité du service après l’exploration de l’essai. Il ne concerne que le service de démonstration.


In [11]:
def submit_proof_of_value(confidence_score: float) -> dict:
    """Envoie une évaluation de qualité après l’exploration de l’essai Agent Module.

    Appelez cette fonction après récupération et application de la logique, afin
    d’indiquer dans quelle mesure les données ont été utiles.

    Args:
        confidence_score: Évaluation de 0.0 (non utile) à 1.0 (très utile).
    """
    # Validation préalable de la présence de la clé d'essai requise pour s'authentifier
    if not _trial_key:
        return {"error": "Aucune clé d’essai disponible. Appelez d’abord get_trial_key."}

    # Transmission du score de valeur (Proof of Value) au point d'accès du service
    return _mcp_call(
        "submit_pov",
        {
            "token": _trial_key,
            "trial_key": _trial_key,
            "confidence_score": confidence_score,
        },
    )

## Lancer un contrôle avec Gemini

La clé d’essai a déjà été obtenue. Nous ne transmettons à Gemini que les deux fonctions d’exécution : `retrieve_compliance_logic` et `submit_proof_of_value`. `get_trial_key` est une initialisation unique ; le modèle ne doit pas la rappeler pendant une session.

Le SDK génère automatiquement les déclarations à partir des types et docstrings. L’instruction système indique ensuite comment employer les outils.


In [18]:
# Déclaration de la liste des outils (fonctions Python) exposés à Gemini
compliance_tools = [retrieve_compliance_logic, submit_proof_of_value]

# Rédaction de la directive système (System Instruction) pour guider le comportement de l'agent :
# - Explique son rôle, sa base de connaissances disponible (le nombre de modules),
# - Définit un protocole d'analyse structuré et une clause obligatoire de non-responsabilité légale.
system_instruction = f"""
Tu es un assistant de pré-analyse de conformité à l’AI Act européen. Tu peux consulter
Agent Module, un service contenant {ai_compliance_module_count} modules structurés.

Lorsqu’on te demande d’évaluer un système :
1. récupère les modules de logique pertinents ;
2. applique les règles au système décrit ;
3. distingue les exigences apparemment satisfaites, les écarts et les informations manquantes ;
4. cite les articles renvoyés par le service ;
5. précise toujours que cette sortie n’est pas un avis juridique ni une certification ;
6. envoie une évaluation de valeur à la fin.

Format de nœud : node:ethics:eth{{NNN}}:logic
Exemples : eth001 (souveraineté des données), eth003 (transparence), eth006 (supervision humaine).
"""

Exécutons maintenant le contrôle. Scénario : un chatbot de service client est déployé dans l’UE, traite des déclarations d’assurance, collecte des données personnelles et produit une première évaluation.

Gemini appellera les fonctions nécessaires, récupérera les modules et générera une pré-analyse structurée.


In [15]:
# Initialisation d'une session de chat avec la configuration des outils et de l'instruction système
chat = client.chats.create(
    model=MODEL_ID,
    config={'tools': compliance_tools, 'system_instruction': system_instruction},
)

# Envoi de la demande d'analyse de conformité pour le chatbot d'assurance
chat.send_message("""
Un chatbot de service client va être déployé dans l’Union européenne. Il traite des
sinistres d’assurance, collecte des données personnelles (nom, numéro de police,
détails du sinistre) et produit une première évaluation des demandes.

Analyse ce système au regard des exigences suivantes :
1. souveraineté et propriété des données (ETH_001) ;
2. transparence et explicabilité (ETH_003) ;
3. supervision humaine (ETH_006).
""")

# Extraction et affichage de l'analyse produite par Gemini tout au long de la conversation
for message in chat.get_history():
    if message.role != 'model':
        continue
    for part in message.parts:
        if part.text:
            print(part.text)

Voici la pré-analyse de conformité à l’AI Act européen pour le chatbot d'assurance, basée sur les modules de logique Agent Module (`ETH_001`, `ETH_003`, `ETH_006`).

---

### Analyse de conformité

#### 1. Souveraineté et propriété des données (`ETH_001`)
*   **Exigences satisfaites :** 
    *   L’utilisation de données personnelles (nom, numéro de police, détails du sinistre) nécessite une base légale claire (RGPD Art. 6) et une minimisation des données collectées (RGPD Art. 25).
*   **Écarts / Risques :** 
    *   Si le chatbot conserve les détails des sinistres ou les données personnelles au-delà du traitement nécessaire sans un consentement explicite et granulaire, il y a violation du principe de limitation de la conservation.
    *   Le transfert éventuel de données vers des services cloud tiers (fournisseurs de LLM/API) non hébergés dans l'EEE sans clauses contractuelles types ou consentement explicite constitue un risque majeur.
*   **Informations manquantes :** 
    *   Où les 

### Examiner l’historique des appels de fonctions

Le SDK a traité automatiquement le flux : Gemini choisit les modules à récupérer, le SDK exécute les fonctions Python puis leurs résultats sont renvoyés au modèle. Inspectons ce qui s’est réellement passé.


In [16]:
from IPython.display import Markdown, display

# Parcours de l'historique complet pour visualiser l'enchaînement logique :
# Permet d'observer à quel moment Gemini a décidé d'appeler les outils externes
for content in chat.get_history():
    display(Markdown(f"### {content.role}:"))
    for part in content.parts:
        # Affichage du texte brut (généralement tronqué si trop long pour l'historique)
        if part.text:
            display(Markdown(part.text[:500]))
        # Indication visuelle si un appel d'outil MCP a été initié par le modèle
        if part.function_call:
            args = dict(part.function_call.args)
            print(f"  Appel de fonction : {part.function_call.name}({args})")
        # Indication visuelle de la réception de la réponse de l'outil par le modèle
        if part.function_response:
            print(f"  Réponse de fonction : {part.function_response.name} -> (tronquée)")
    print("-" * 80)

### user:


Un chatbot de service client va être déployé dans l’Union européenne. Il traite des
sinistres d’assurance, collecte des données personnelles (nom, numéro de police,
détails du sinistre) et produit une première évaluation des demandes.

Analyse ce système au regard des exigences suivantes :
1. souveraineté et propriété des données (ETH_001) ;
2. transparence et explicabilité (ETH_003) ;
3. supervision humaine (ETH_006).


--------------------------------------------------------------------------------


### model:

  Appel de fonction : retrieve_compliance_logic({'vertical': 'ai-compliance', 'node_id': 'node:ethics:eth001:logic'})
--------------------------------------------------------------------------------


...

### user:

  Réponse de fonction : submit_proof_of_value -> (tronquée)
--------------------------------------------------------------------------------


### model:

Voici la pré-analyse de conformité à l’AI Act européen pour le chatbot d'assurance, basée sur les modules de logique Agent Module (`ETH_001`, `ETH_003`, `ETH_006`).

---

### Analyse de conformité

#### 1. Souveraineté et propriété des données (`ETH_001`)
*   **Exigences satisfaites :** 
    *   L’utilisation de données personnelles (nom, numéro de police, détails du sinistre) nécessite une base légale claire (RGPD Art. 6) et une minimisation des données collectées (RGPD Art. 25).
*   **Écarts /

--------------------------------------------------------------------------------


## Inspecter la logique brute

Vous pouvez aussi appeler directement Agent Module afin d’examiner les données de conformité transmises à Gemini. Chaque nœud contient des règles déterministes et des références réglementaires.


In [17]:
# Test d'interrogation brute : Récupération manuelle et affichage complet du module de transparence (ETH_003)
transparency = retrieve_compliance_logic("ethics", "node:ethics:eth003:logic")

# Affichage structuré du JSON de réponse pour analyse technique
print(json.dumps(transparency, indent=4))

{
    "jsonrpc": "2.0",
    "id": 1,
    "result": {
        "content": [
            {
                "type": "text",
                "text": "{\n  \"object\": \"traversal_response\",\n  \"vertical\": \"ethics\",\n  \"node_id\": \"node:ethics:eth003:logic\",\n  \"layer\": \"logic\",\n  \"access\": \"trial\",\n  \"content\": {\n    \"node_id\": \"ETH_003_TRANSPARENCY_logic\",\n    \"vertical\": \"ethics\",\n    \"version\": \"2.0.0\",\n    \"last_validated\": \"2026-02-28\",\n    \"wikidata_ref\": \"https://www.wikidata.org/wiki/Q535946\",\n    \"eu_ai_act_articles\": [\n      \"Art. 13\",\n      \"Art. 17\"\n    ],\n    \"weighting\": \"Essential\",\n    \"subject\": \"algorithmic_transparency_and_logging\",\n    \"description\": \"Deterministic rules governing transparency obligations for AI systems \u2014 human-readable logging, decision trail auditing, tamper-evident storage, and EU AI Act Art. 13 information disclosure requirements. Transparency is a legal obligation for high-ris

## Expériences à réaliser

- **Autres modules :** interrogez la détection des biais (ETH_008), la confidentialité (ETH_010) ou la classification à haut risque (ETH_015).
- **Autres scénarios :** testez un assistant de diagnostic médical, un outil de présélection de candidatures ou une modération de contenus.
- **Couches différentes :** remplacez `logic` par `directive` dans l’identifiant pour récupérer des garde-fous procéduraux plutôt que des critères réussite/échec.
- **Récupérations parallèles :** demandez cinq modules et observez si Gemini émet des appels parallèles.
- **Forcer les outils :** selon la version du SDK, ajoutez une configuration de function calling pour exiger un appel d’outil.

Dans chaque cas, identifiez ce qui vient du service externe, ce qui relève de l’interprétation du modèle et ce qui devrait faire l’objet d’une validation humaine.


## Découvrir les modules disponibles

La verticale de conformité IA d’Agent Module couvre différentes parties de l’AI Act. Un module peut présenter quatre couches : **logic** (critères réussite/échec), **directive** (garde-fous procéduraux), **skill** (connaissance métier) et **action** (gabarits exécutables).

La bibliothèque évolue. Pour consulter son contenu à l’exécution plutôt que de figer une liste :

- `GET https://api.agent-module.dev/api/status` : nombre actuel de modules et métadonnées ;
- `GET https://api.agent-module.dev/api/demo?vertical=ai-compliance` : manifeste avec identifiants, titres et liens aux articles ;
- `GET https://api.agent-module.dev/llms.txt` : index lisible par un agent.

Ainsi, l’agent accède à la bibliothèque active plutôt qu’à un instantané.


## Étapes suivantes

### Références utiles

- [Dépôt MCP d’Agent Module](https://github.com/AgentModule/mcp) : documentation des outils et installation.
- [Spécification OpenAPI d’Agent Module](https://agent-module.dev/openapi.json) : import dans Vertex AI Agent Builder.
- [Documentation Gemini Function Calling](https://ai.google.dev/gemini-api/docs/function-calling).
- [Texte de l’AI Act](https://artificialintelligenceact.eu/).

### Exemples associés

- [Function calling](../quickstarts/Function_calling.ipynb) : les fondamentaux.
- [Barista Bot](../examples/Agents_Function_Calling_Barista_Bot.ipynb) : agent de commande avec fonctions.
- [Browser as a tool](../examples/Browser_as_a_tool.ipynb) : intégration d’un outil externe.

### Poursuivre avec Gemini

- Explorez la configuration des appels de fonctions dans [Function calling config](../quickstarts/Function_calling_config.ipynb).
- Utilisez [JSON mode](../quickstarts/JSON_mode.ipynb) pour les sorties structurées.
- Essayez la [Live API avec outils](../quickstarts/Get_started_LiveAPI_tools.ipynb) pour les appels temps réel.
